# View data struct

In [5]:
# Step 0: 先探索文件结构，搞清楚里面有什么
import anndata as ad
import numpy as np

data_path = "/home/liyang/BioWuYan/dygmamba_project/data/hemta/BMMC/"
data_file = data_path + "GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad"
adata = ad.read_h5ad( data_file)

print(adata)
print("\n--- obs columns (细胞元信息) ---")
print(adata.obs.columns.tolist())
print("\n--- var columns (基因/peak元信息) ---")
print(adata.var.columns.tolist())
print("\n--- obsm keys ---")
print(list(adata.obsm.keys()))
print("\n--- uns keys ---")
print(list(adata.uns.keys()))
print("\n--- layers ---")
print(list(adata.layers.keys()))

# 检查模态标记
print("\n--- 模态分布 ---")
if 'feature_types' in adata.var.columns:
    print(adata.var['feature_types'].value_counts())
elif 'modality' in adata.var.columns:
    print(adata.var['modality'].value_counts())

AnnData object with n_obs × n_vars = 69249 × 129921
    obs: 'GEX_pct_counts_mt', 'GEX_n_counts', 'GEX_n_genes', 'GEX_size_factors', 'GEX_phase', 'ATAC_nCount_peaks', 'ATAC_atac_fragments', 'ATAC_reads_in_peaks_frac', 'ATAC_blacklist_fraction', 'ATAC_nucleosome_signal', 'cell_type', 'batch', 'ATAC_pseudotime_order', 'GEX_pseudotime_order', 'Samplename', 'Site', 'DonorNumber', 'Modality', 'VendorLot', 'DonorID', 'DonorAge', 'DonorBMI', 'DonorBloodType', 'DonorRace', 'Ethnicity', 'DonorGender', 'QCMeds', 'DonorSmoker'
    var: 'feature_types', 'gene_id'
    uns: 'ATAC_gene_activity_var_names', 'dataset_id', 'genome', 'organism'
    obsm: 'ATAC_gene_activity', 'ATAC_lsi_full', 'ATAC_lsi_red', 'ATAC_umap', 'GEX_X_pca', 'GEX_X_umap'
    layers: 'counts'

--- obs columns (细胞元信息) ---
['GEX_pct_counts_mt', 'GEX_n_counts', 'GEX_n_genes', 'GEX_size_factors', 'GEX_phase', 'ATAC_nCount_peaks', 'ATAC_atac_fragments', 'ATAC_reads_in_peaks_frac', 'ATAC_blacklist_fraction', 'ATAC_nucleosome_signal', '

# Read and split RNA & ATAC

In [6]:
# bmmc_preprocess.py
# 将 GSE194122 处理成你的 DyGMamba pipeline 所需的全部文件

import os
import sys
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp
import scipy.io as sio
import pickle
from scipy import sparse
sys.path.append('/home/liyang/BioWuYan/dygmamba_project/model/dygmamba/src/')
# ============================================================
# 配置路径
# ============================================================

RAW_H5AD = data_path + "GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad"
OUTPUT_PATH = data_path + "process/"
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH + "R/rna/", exist_ok=True)
os.makedirs(OUTPUT_PATH + "R/atac/", exist_ok=True)

# ============================================================
# Step 1: 读取并拆分 RNA / ATAC
# ============================================================
print("=== Step 1: 读取原始文件 ===")
adata_all = ad.read_h5ad(RAW_H5AD)
print(adata_all)

# 确认模态列名（二选一）
if 'feature_types' in adata_all.var.columns:
    modality_col = 'feature_types'
    rna_label  = 'GEX'
    atac_label = 'ATAC'
elif 'modality' in adata_all.var.columns:
    modality_col = 'modality'
    rna_label  = 'Gene Expression'
    atac_label = 'Peaks'
else:
    raise ValueError("找不到模态列，请先检查 adata_all.var.columns")

# 拆分
rna_mask  = adata_all.var[modality_col] == rna_label
atac_mask = adata_all.var[modality_col] == atac_label

adata_rna  = adata_all[:, rna_mask].copy()
adata_atac = adata_all[:, atac_mask].copy()

print(f"RNA  shape: {adata_rna.shape}")   # 应为 ~69249 × 13431
print(f"ATAC shape: {adata_atac.shape}")  # 应为 ~69249 × 116490

# ============================================================
# Step 2: 处理 RNA
# ============================================================
print("\n=== Step 2: RNA 预处理 ===")

# 保存原始counts（你的pipeline需要 layers['counts']）
if 'counts' not in adata_rna.layers:
    # 这个数据集X本身已是normalized，需要找raw counts
    if adata_rna.raw is not None:
        adata_rna.layers['counts'] = adata_rna.raw.X[:, 
            adata_rna.raw.var_names.isin(adata_rna.var_names)].copy()
    else:
        # 如果没有raw，用X作为counts（已经是log-normalized）
        adata_rna.layers['counts'] = adata_rna.X.copy()

# 标准化（如果X不是log-normalized）
# 这个数据集X已经是log-normalized，跳过normalize步骤
# 只做 HVG 和 PCA
sc.pp.highly_variable_genes(adata_rna, n_top_genes=5000, 
                             flavor='seurat_v3', layer='counts')

sc.pp.pca(adata_rna, n_comps=50, use_highly_variable=True)
sc.pp.neighbors(adata_rna, n_neighbors=15)
sc.tl.umap(adata_rna)

# 伪时间计算（DPT）
# 选 HSC 作为根细胞（cell_type 列根据实际情况调整）
print("\n细胞类型分布：")
if 'cell_type' in adata_rna.obs.columns:
    print(adata_rna.obs['cell_type'].value_counts().head(10))
    hsc_mask = adata_rna.obs['cell_type'].str.contains('HSC|hematopoietic stem', 
                                                         case=False, na=False)
    if hsc_mask.sum() > 0:
        adata_rna.uns['iroot'] = np.where(hsc_mask)[0][0]
        sc.tl.diffmap(adata_rna, n_comps=10)
        sc.tl.dpt(adata_rna)
        print(f"伪时间计算完成，HSC根细胞索引: {adata_rna.uns['iroot']}")
    else:
        print("未找到HSC，用第一个细胞作为根")
        adata_rna.uns['iroot'] = 0
        sc.tl.diffmap(adata_rna, n_comps=10)
        sc.tl.dpt(adata_rna)
elif 'cell_type' not in adata_rna.obs.columns:
    # 尝试其他可能的列名
    for col in ['celltype', 'leiden', 'louvain', 'cluster']:
        if col in adata_rna.obs.columns:
            print(f"使用 {col} 列作为细胞类型")
            adata_rna.obs['cell_type'] = adata_rna.obs[col]
            break

# 保存 gene_info（你的pipeline需要基因坐标信息）
# 从 var 中提取，或用 GTF 文件补充
gene_info = adata_rna.var.copy()
if 'chrom' not in gene_info.columns:
    # 如果没有坐标信息，需要从GTF获取（见下方 Step 5）
    print("警告：var中没有坐标信息，需要从GTF补充（见Step 5）")

adata_rna.write_h5ad(OUTPUT_PATH + "rna_processed.h5ad")
print(f"RNA保存: {OUTPUT_PATH}rna_processed.h5ad")

# 保存R格式
cellinfo = adata_rna.obs
rnainfo  = adata_rna.var
mtx      = adata_rna.layers['counts']
cellinfo.to_csv(OUTPUT_PATH + "R/rna/cellinfo.csv")
rnainfo.to_csv(OUTPUT_PATH + "R/rna/rnainfo.csv")
sio.mmwrite(OUTPUT_PATH + "R/rna/sparse.mtx", mtx)


# ============================================================
# Step 3: 处理 ATAC
# ============================================================
print("\n=== Step 3: ATAC 预处理 ===")

# 统一peak名称格式为 chr1-1000-2000（你的pipeline期望的格式）
def normalize_peak_names(var_names):
    """统一peak格式: chr1:1000-2000 → chr1-1000-2000"""
    new_names = []
    for name in var_names:
        name = str(name)
        # 处理 chr1:1000-2000 格式
        if ':' in name and '-' in name:
            name = name.replace(':', '-', 1)
        # 处理 chr1_1000_2000 格式  
        elif name.count('_') >= 2:
            parts = name.split('_', 1)
            if parts[0].startswith('chr'):
                name = parts[0] + '-' + parts[1].replace('_', '-')
        new_names.append(name)
    return new_names

adata_atac.var_names = normalize_peak_names(adata_atac.var_names)

# 使用你已有的 filter_atac_base 函数
import sys
sys.path.append('/path/to/your/project/src')  # 修改为你的项目路径
from pdata.data_preprocess import filter_atac_base

adata_atac = filter_atac_base(adata_atac)
print(f"过滤后 ATAC shape: {adata_atac.shape}")

# TF-IDF 归一化（scATAC 标准做法）
import muon.atac as ma
ma.pp.tfidf(adata_atac, scale_factor=1e4)

# SVD降维
sc.pp.pca(adata_atac, n_comps=50)
sc.pp.neighbors(adata_atac)
sc.tl.umap(adata_atac)

adata_atac.write_h5ad(OUTPUT_PATH + "atac_processed.h5ad")
print(f"ATAC保存: {OUTPUT_PATH}atac_processed.h5ad")

# 保存R格式
cellinfo = adata_atac.obs
atacinfo = adata_atac.var
mtx      = adata_atac.X
cellinfo.to_csv(OUTPUT_PATH + "R/atac/cellinfo.csv")
atacinfo.to_csv(OUTPUT_PATH + "R/atac/atacinfo.csv")
sio.mmwrite(OUTPUT_PATH + "R/atac/sparse.mtx", mtx)

=== Step 1: 读取原始文件 ===
AnnData object with n_obs × n_vars = 69249 × 129921
    obs: 'GEX_pct_counts_mt', 'GEX_n_counts', 'GEX_n_genes', 'GEX_size_factors', 'GEX_phase', 'ATAC_nCount_peaks', 'ATAC_atac_fragments', 'ATAC_reads_in_peaks_frac', 'ATAC_blacklist_fraction', 'ATAC_nucleosome_signal', 'cell_type', 'batch', 'ATAC_pseudotime_order', 'GEX_pseudotime_order', 'Samplename', 'Site', 'DonorNumber', 'Modality', 'VendorLot', 'DonorID', 'DonorAge', 'DonorBMI', 'DonorBloodType', 'DonorRace', 'Ethnicity', 'DonorGender', 'QCMeds', 'DonorSmoker'
    var: 'feature_types', 'gene_id'
    uns: 'ATAC_gene_activity_var_names', 'dataset_id', 'genome', 'organism'
    obsm: 'ATAC_gene_activity', 'ATAC_lsi_full', 'ATAC_lsi_red', 'ATAC_umap', 'GEX_X_pca', 'GEX_X_umap'
    layers: 'counts'
RNA  shape: (69249, 13431)
ATAC shape: (69249, 116490)

=== Step 2: RNA 预处理 ===


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/scanpy/preprocessing/_pca.py:377: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  warn(msg, FutureWarning)



细胞类型分布：
cell_type
CD8+ T              11589
CD14+ Mono          10843
NK                   6929
CD4+ T activated     5526
Naive CD20+ B        5052
Erythroblast         4916
CD4+ T naive         4398
Transitional B       2810
Proerythroblast      2300
CD16+ Mono           1894
Name: count, dtype: int64
伪时间计算完成，HSC根细胞索引: 108
警告：var中没有坐标信息，需要从GTF补充（见Step 5）
RNA保存: /home/liyang/BioWuYan/dygmamba_project/data/hemta/BMMC/process/rna_processed.h5ad

=== Step 3: ATAC 预处理 ===
初始 Peak 数量: 116490
过滤后 ATAC shape: (69249, 116468)
ATAC保存: /home/liyang/BioWuYan/dygmamba_project/data/hemta/BMMC/process/atac_processed.h5ad


# Gene info

In [7]:
# ============================================================
# Step 4: 生成 gene_info_data.pkl（基因坐标，pipeline必须）
# ============================================================
print("\n=== Step 4: 生成 gene_info ===")


# 方法B：从GTF文件解析（需要先下载hg38 GTF）
# wget https://ftp.ensembl.org/pub/release-109/gtf/homo_sapiens/Homo_sapiens.GRCh38.109.gtf.gz
from gtfparse import read_gtf
import pandas as pd

gtf_file = data_path + "Homo_sapiens.GRCh38.109.gtf.gz"
gtf = read_gtf(gtf_file)

# ---- 关键修复：判断类型，统一转成 pandas ----
import polars as pl
if isinstance(gtf, pl.DataFrame):
    # Polars 用 filter，不能用布尔 mask 直接切片
    genes = gtf.filter(pl.col("feature") == "gene")
    gene_info_df = genes.select(["gene_name", "seqname", "start", "end", "strand"]).to_pandas()
else:
    # 旧版 gtfparse 返回 pandas，原来的写法
    genes = gtf[gtf["feature"] == "gene"]
    gene_info_df = genes[["gene_name", "seqname", "start", "end", "strand"]].copy()

# 后续统一用 pandas 操作
gene_info_df.columns = ["gene_name", "chrom", "start", "end", "strand"]
gene_info_df["chrom"] = "chr" + gene_info_df["chrom"].astype(str)

# 只保留标准染色体
standard_chroms = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]
gene_info_df = gene_info_df[gene_info_df["chrom"].isin(standard_chroms)]

# 去重（同一基因可能有多行）
gene_info_df = gene_info_df.drop_duplicates(subset=["gene_name"]).reset_index(drop=True)

# 只保留 RNA 里有的基因
rna_genes = set(adata_rna.var_names)
gene_info_df = gene_info_df[gene_info_df["gene_name"].isin(rna_genes)]

gene_info_df.to_pickle(OUTPUT_PATH + "gene_info_data.pkl")
gene_info_df.to_pickle(OUTPUT_PATH + "gene_info_filtered.pkl")
print(f"gene_info 保存完成：{len(gene_info_df)} 个基因")
print(gene_info_df.head())


=== Step 4: 生成 gene_info ===


INFO:root:Extracted GTF attributes: ['gene_id', 'gene_version', 'gene_name', 'gene_source', 'gene_biotype', 'transcript_id', 'transcript_version', 'transcript_name', 'transcript_source', 'transcript_biotype', 'tag', 'ccds_id', 'exon_number', 'exon_id', 'exon_version', 'protein_id', 'protein_version', 'transcript_support_level']


gene_info 保存完成：12097 个基因
   gene_name chrom     start       end strand
0     ATAD3B  chr1   1471765   1497848      +
4      PEX10  chr1   2403964   2413797      -
7      PEX14  chr1  10472288  10630758      +
11     PLCH2  chr1   2425980   2505532      +
12     SPSB1  chr1   9292894   9369532      +


# fimo scan

In [8]:
# ============================================================
# Step 5: 生成 JASPAR motif 数据（jaspar_df.pkl）
# ============================================================
print("\n=== Step 5: JASPAR motif 扫描 ===")
import anndata as ad

adata_atac = ad.read_h5ad(data_path + "process/atac_processed.h5ad")
peaks = adata_atac.var_names.tolist()

bed_lines = []
for p in peaks:
    # 统一格式：chr1:1000-2000 或 chr1-1000-2000 都处理
    p_clean = p.replace(':', '-')
    parts = p_clean.split('-')
    if len(parts) == 3:
        chrom, start, end = parts[0], parts[1], parts[2]
        # 注意：必须用 \t 分隔，不能用空格
        bed_lines.append(f"{chrom}\t{start}\t{end}\t{p_clean}")

output_bed = "peaks.bed"
with open(output_bed, 'w') as f:
    f.write('\n'.join(bed_lines) + '\n')

print(f"写出 {len(bed_lines)} 个peaks")

# 验证前5行格式
with open(output_bed) as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        cols = line.rstrip('\n').split('\t')
        print(f"行{i+1}: {len(cols)} 列 → {cols}")


# ==================================
# 运行 fimo_bash.sh


=== Step 5: JASPAR motif 扫描 ===
写出 116468 个peaks
行1: 4 列 → ['chr1', '9776', '10668', 'chr1-9776-10668']
行2: 4 列 → ['chr1', '180726', '181005', 'chr1-180726-181005']
行3: 4 列 → ['chr1', '181117', '181803', 'chr1-181117-181803']
行4: 4 列 → ['chr1', '191133', '192055', 'chr1-191133-192055']
行5: 4 列 → ['chr1', '267562', '268456', 'chr1-267562-268456']


In [ ]:
# FIMO完成后，读取结果并转换成你pipeline要求的格式
import pandas as pd

def parse_fimo_to_jaspar_pkl(fimo_tsv_path, output_pkl_path):
    """
    将FIMO输出转换成 pipeline 期望的 jaspar_df.pkl 格式
    """
    print(f"读取FIMO结果: {fimo_tsv_path}")
    fimo_df = pd.read_csv(fimo_tsv_path, sep='\t', comment='#')
    # fimo输出列: motif_id, motif_alt_id, sequence_name, start, stop, strand, score, p-value, q-value, matched_sequence
    
    # sequence_name 就是你的peak名
    # motif_alt_id 通常是TF名称（如 MA0139.1::CTCF → 取CTCF部分）
    fimo_df['TF_Symbol'] = fimo_df['motif_alt_id'].str.split('::').str[-1]
    
    # 过滤低质量hits
    fimo_df = fimo_df[fimo_df['q-value'] < 0.05]
    
    jaspar_df = fimo_df[['sequence_name', 'TF_Symbol', 'score']].copy()
    jaspar_df.columns = ['sequence_name', 'TF_Symbol', 'score']
    
    jaspar_df.to_pickle(output_pkl_path)
    print(f"jaspar_df.pkl 保存: {output_pkl_path}")
    print(f"  包含 {jaspar_df['TF_Symbol'].nunique()} 个TF, {jaspar_df['sequence_name'].nunique()} 个peaks")
    
    return jaspar_df

# 等FIMO跑完后执行:

data_path = "/home/liyang/BioWuYan/dygmamba_project/data/hemta/BMMC/"

jaspar_df = parse_fimo_to_jaspar_pkl(data_path + "fimo_output/fimo.tsv", data_path + "jaspar_df.pkl")


# ChIP-seq

In [ ]:
# ============================================================
# Step 6: ChIP-seq数据（替代方案：ENCODE血液细胞ChIP-seq）
# ============================================================
print("\n=== Step 6: 下载血液细胞 ChIP-seq ===")

# 运行 get_data.sh 下载 ENCODE 血液细胞 ChIP-seq 数据

# RP score

In [ ]:
from pdata.data_preprocess import (calculate_rp_250kb, calculate_peak_peak_rp,
                                    analyze_score_distribution, find_zero_sum_elements)
import anndata as ad

output_path = data_path + "process/"
adata_rna  = ad.read_h5ad(output_path + "rna_processed.h5ad")
adata_atac = ad.read_h5ad(output_path + "atac_processed.h5ad")
gene_info  = pd.read_pickle(output_path + "gene_info_filtered.pkl")

# Peak-Gene RP Score（250kb窗口，50kb衰减）
adata_rp = calculate_rp_250kb(
    adata_atac=adata_atac, 
    adata_rna=adata_rna,
    gene_info_df=gene_info,
    decay_dist=50000,
    max_range=250000
)

# Peak-Peak RP Score
adata_peak_rp = calculate_peak_peak_rp(
    adata_atac, decay_distance=50000, max_range=250000
)

# 二值化
rp_stat = analyze_score_distribution(adata_rp)
rp_threshold = 0  # 或用 rp_stat["q0.75"]

adata_rp_binary = adata_rp.copy()
adata_rp_binary.X = (adata_rp_binary.X > rp_threshold).astype(int)

adata_peak_rp_binary = adata_peak_rp.copy()
adata_peak_rp_binary.X = (adata_peak_rp_binary.X > 0).astype(int)

# 清理全零元素
zero_peaks, zero_genes = find_zero_sum_elements(adata_rp_binary)
if zero_peaks:
    adata_rp = adata_rp[:, ~adata_rp.var_names.isin(zero_peaks)]
    adata_rp_binary = adata_rp_binary[:, ~adata_rp_binary.var_names.isin(zero_peaks)]
    adata_atac = adata_atac[:, ~adata_atac.var_names.isin(zero_peaks)]

# 保存
adata_rp.write_h5ad(output_path + "peak_gene_rp_network.h5ad")
adata_peak_rp.write_h5ad(output_path + "peak_peak_rp_network.h5ad")
adata_rp_binary.write_h5ad(output_path + "binary_peak_gene_rp_network.h5ad")
adata_peak_rp_binary.write_h5ad(output_path + "binary_peak_peak_rp_network.h5ad")
adata_atac.write_h5ad(output_path + "atac_processed.h5ad")  # 更新过滤后版本
print("RP网络计算完成！")

# DygMamba

In [1]:

import os
import sys
import subprocess

import networkx as nx
import scanpy as sc
import pandas as pd
import anndata as ad
import pickle
import code
import scipy.sparse as sp
import numpy as np
import h5py


import matplotlib.pyplot as plt
import scipy.io
from scipy.io import mmread

from pkg_resources import resource_filename
from datetime import datetime
from collections import Counter

# ************************** self function ******************

sys.path.append("/home/wuyan/dygmamba_project/model/dygmamba/src")

In [2]:
root_path = "/home/wuyan/dygmamba_project/data/hemta/BMMC/"
    
data_path = root_path +"process/"

adata_rna = ad.read_h5ad(data_path + "rna_processed.h5ad")

adata_atac = ad.read_h5ad(data_path + "atac_processed.h5ad")

pseudotime = pd.read_csv(data_path + "max_cells_lineage_pseudotime.csv")

In [5]:
set(adata_rna.obs['cell_type'])

{'B1 B',
 'CD14+ Mono',
 'CD16+ Mono',
 'CD4+ T activated',
 'CD4+ T naive',
 'CD8+ T',
 'CD8+ T naive',
 'Erythroblast',
 'G/M prog',
 'HSC',
 'ID2-hi myeloid prog',
 'ILC',
 'Lymph prog',
 'MK/E prog',
 'NK',
 'Naive CD20+ B',
 'Plasma cell',
 'Transitional B',
 'cDC2',
 'pDC'}